# Experiment 4: ORDER_LINE ⋈ STOCK (HTAP Join Validation)

Stacked bar chart (fixed update intensity) + line chart (latency vs update intensity).  
Configurations: **SNAP, MONO-NR/RR/WR, DUAL-WR, EPOCH-NR/RR/WR** with Zipfian skewed updates.

In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pandas', 'matplotlib'])
print('done')

In [ ]:
from pathlib import Path
import subprocess, os, sys
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np

ROOT      = Path('../../').resolve()
EXP4_DIR  = (ROOT / 'benches' / 'exp4_stock_join').resolve()
DATA_DIR  = EXP4_DIR / 'data'
FIGS_DIR  = EXP4_DIR / 'figs'

FIGS_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

# ===== CONFIG =====
WAREHOUSES   = 1
DIST         = 'zipf0.99'
ZIPF_THETA   = 0.99
BUCKET_NUM   = 1024
WARMUP       = 1
REPEAT       = 10
TRIM         = 2
UPDATE_PCTS  = [0, 5, 10, 15, 20]
MAIN_PCT     = 10   # fixed intensity for stacked bar chart

# All series: (table_type, repair_mode, label)
SERIES = [
    ('naive', 'nr', 'SNAP'),
    ('heap',  'nr', 'MONO-NR'),
    ('heap',  'rr', 'MONO-RR'),
    ('heap',  'wr', 'MONO-WR'),
    ('chain', 'wr', 'DUAL-WR'),
    ('par',   'nr', 'EPOCH-NR'),
    ('par',   'rr', 'EPOCH-RR'),
    ('par',   'wr', 'EPOCH-WR'),
]
# ==================

W             = WAREHOUSES
BIN           = ROOT / 'target' / 'release' / 'stock_join_bench'
STOCK_FILE    = DATA_DIR / f'stock_w{W}.tbl'
OL_FILE       = DATA_DIR / f'orderline_w{W}.tbl'
OUT_CSV       = DATA_DIR / f'exp4_w{W}_{DIST}.csv'

print('ROOT    :', ROOT)
print('CSV     :', OUT_CSV)

In [ ]:
# ── Build Rust binary ──────────────────────────────────────────────────────
print('Building stock_join_bench...')
result = subprocess.run(
    ['cargo', 'build', '--release', '--bin', 'stock_join_bench'],
    cwd=ROOT, capture_output=True, text=True,
)
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])
    raise RuntimeError('cargo build failed')
print('Build OK')

In [ ]:
# ── Generate CH-benCHmark data (if not present) ────────────────────────────
if not STOCK_FILE.exists() or not OL_FILE.exists():
    print(f'Generating CH-benCHmark data (W={W})...')
    subprocess.run([
        sys.executable, str(EXP4_DIR / 'generate_chbenchmark_data.py'),
        '--warehouses', str(W),
        '--output-dir', str(DATA_DIR),
    ], check=True)
else:
    print(f'Data files exist: {STOCK_FILE.name}, {OL_FILE.name}')

stock_rows = sum(1 for _ in open(STOCK_FILE))
ol_rows    = sum(1 for _ in open(OL_FILE))
print(f'STOCK: {stock_rows:,}  ORDER_LINE: {ol_rows:,}')

In [ ]:
# ── Generate update files ──────────────────────────────────────────────────
for pct in UPDATE_PCTS:
    uf = DATA_DIR / f'stock_updates_w{W}_{pct}pct_{DIST}.tbl'
    if uf.exists():
        print(f'  {pct}% exists, skipping.')
        continue
    if pct == 0:
        uf.touch()
        print(f'  0% -> empty file created.')
    else:
        subprocess.run([
            sys.executable, str(EXP4_DIR / 'generate_stock_updates.py'),
            str(STOCK_FILE), str(pct), str(W),
            '--zipf', str(ZIPF_THETA),
            '--output-dir', str(DATA_DIR),
        ], check=True)
        print(f'  {pct}% -> {uf.name}')

In [ ]:
# ── Run benchmark (all configs × all update intensities) ───────────────────
if OUT_CSV.exists():
    OUT_CSV.unlink()
    print('Removed old CSV')

total_runs = len(SERIES) * len(UPDATE_PCTS)
current    = 0

for pct in UPDATE_PCTS:
    uf = DATA_DIR / f'stock_updates_w{W}_{pct}pct_{DIST}.tbl'
    for (ttype, rmode, label) in SERIES:
        current += 1
        print(f'[{current}/{total_runs}] {label} | update={pct}% | {DIST}')
        cmd = [
            str(BIN),
            '--stock-file',     str(STOCK_FILE),
            '--orderline-file', str(OL_FILE),
            '--updates-file',   str(uf),
            '--table-type',     ttype,
            '--repair-mode',    rmode,
            '--bucket-num',     str(BUCKET_NUM),
            '--warmup',         str(WARMUP),
            '--repeat',         str(REPEAT),
            '--trim',           str(TRIM),
            '--update-pct',     str(pct),
            '--distribution',   DIST,
            '--output-csv',     str(OUT_CSV),
        ]
        r = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True)
        if r.returncode != 0:
            print(f'  FAILED: {r.stderr[:500]}')
        else:
            for line in r.stdout.strip().split('\n'):
                if 'total_ms' in line or 'trimmed' in line:
                    print(f'  {line.strip()}')

print(f'\nDone. Results -> {OUT_CSV}')

In [ ]:
# ── Load results ──────────────────────────────────────────────────────────
df = pd.read_csv(OUT_CSV)

table_map  = {'Heap': 'heap', 'Chain': 'chain', 'Par': 'par', 'Naive': 'naive'}
repair_map = {'Nr': 'NR', 'Rr': 'RR', 'Wr': 'WR'}
df['table']  = df['table_type'].map(table_map).fillna(df['table_type'])
df['repair'] = df['repair_mode'].map(repair_map).fillna(df['repair_mode'])
df.loc[df['table'] == 'naive', 'repair'] = ''

# Derive total_ms if not present
if 'total_ms' not in df.columns:
    df['total_ms'] = (
        df['join1_build_ms'] + df['join1_probe_ms'] +
        df['update_ms'] + df['join2_build_ms'] + df['join2_probe_ms']
    )

display_cols = [
    'table', 'repair', 'update_pct',
    'j1_alloc_ms', 'j1_insert_ms', 'join1_probe_ms',
    'update_ms', 'join2_build_ms', 'join2_probe_ms', 'total_ms'
]
display(df[df['update_pct'] == MAIN_PCT][display_cols])

In [ ]:
# ── Figure 1: Stacked bar chart at fixed update intensity (MAIN_PCT) ───────
#    Same style as Q14 notebook (Paul Tol Bright palette, solid=write, hatch=read)

tol = {
    'blue':   '#4477AA',
    'cyan':   '#66CCEE',
    'green':  '#228833',
    'yellow': '#CCBB44',
    'red':    '#EE6677',
    'purple': '#AA3377',
    'grey':   '#BBBBBB',
}

phases = [
    ('j1_alloc_ms',  'J1 Alloc',  tol['cyan'],   True),
    ('j1_insert_ms', 'J1 Insert', tol['blue'],   True),
    ('join1_probe_ms','J1 Probe', tol['red'],    False),
    ('update_ms',    'Update',    tol['yellow'], True),
    ('join2_build_ms','J2 Build', tol['purple'], True),
    ('join2_probe_ms','J2 Probe', tol['green'],  False),
]

hatch_map = {'J1 Probe': '///', 'J2 Probe': '\\\\\\'}

table_order   = ['naive', 'heap', 'chain', 'par']
table_display = {'naive': 'SNAP', 'heap': 'MONO', 'chain': 'DUAL', 'par': 'EPOCH'}
repair_order  = {'naive': [''], 'heap': ['NR','RR','WR'], 'chain': ['WR'], 'par': ['NR','RR','WR']}

plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif']  = ['Times New Roman', 'Times', 'Nimbus Roman No9 L', 'DejaVu Serif']
plt.rcParams['hatch.linewidth'] = 0.9

sub = df[df['update_pct'] == MAIN_PCT].copy()

bar_w = 0.58
gap   = 0.36
bar_x, minor_labels, major_centers, major_labels_list, combos = [], [], [], [], []
x = 0.0
for t in table_order:
    subs  = repair_order[t]
    start = x
    for r in subs:
        bar_x.append(x)
        minor_labels.append(r)
        combos.append((t, r))
        x += 0.78
    end = x - 0.78
    major_centers.append((start + end) / 2.0)
    major_labels_list.append(table_display[t])
    x += gap

fig, ax = plt.subplots(figsize=(7.8, 6.0))
fig.subplots_adjust(bottom=0.22)

for i, (t, r) in enumerate(combos):
    row = sub[(sub['table'] == t) & (sub['repair'] == r)]
    if row.empty:
        continue
    row = row.iloc[0]
    bottom = 0.0
    for (col, lbl, color, is_write) in phases:
        v = float(row.get(col, 0.0))
        if v <= 0:
            continue
        if is_write:
            ax.bar(bar_x[i], v, bottom=bottom, width=bar_w,
                   color=color, edgecolor='black', linewidth=0.4, zorder=2)
        else:
            ax.bar(bar_x[i], v, bottom=bottom, width=bar_w,
                   color='white', edgecolor='black', linewidth=0.4, zorder=2)
            ax.bar(bar_x[i], v, bottom=bottom, width=bar_w,
                   color='none', edgecolor=color, linewidth=0.9,
                   hatch=hatch_map.get(lbl, '///'), zorder=3)
        bottom += v

ax.set_xticks([])
for xi, lab in zip(bar_x, minor_labels):
    ax.text(xi, -0.028, lab, ha='center', va='top',
            transform=ax.get_xaxis_transform(), fontsize=10, clip_on=False)
for xc, lab in zip(major_centers, major_labels_list):
    ax.text(xc, -0.072, lab, ha='center', va='top',
            transform=ax.get_xaxis_transform(), fontsize=12, clip_on=False)

for k in range(1, len(major_centers)):
    prev_end   = max(xx for xx, (tt, _) in zip(bar_x, combos) if tt == table_order[k-1])
    next_start = min(xx for xx, (tt, _) in zip(bar_x, combos) if tt == table_order[k])
    ax.axvline((prev_end + next_start) / 2, linestyle=':', linewidth=0.6, alpha=0.3)

ax.set_axisbelow(True)
ax.yaxis.grid(True, linestyle='--', linewidth=0.6, alpha=0.6)
ax.set_ylabel('Duration (ms)')
ax.set_title(f'Exp 4: ORDER_LINE ⋈ STOCK (W={W}, {MAIN_PCT}% Zipfian update)')

handles, labels_leg = [], []
for (col, lbl, color, is_write) in reversed(phases):
    if is_write:
        patch = Patch(facecolor=color, edgecolor='black', linewidth=0.4)
    else:
        patch = Patch(facecolor='white', edgecolor=color,
                      hatch=hatch_map.get(lbl, '///'), linewidth=0.9)
    handles.append(patch)
    labels_leg.append(lbl)

ax.legend(handles, labels_leg, title='Operations',
          loc='upper right', framealpha=0.9, fontsize=8.5, title_fontsize=8.5)

plt.tight_layout()
out_pdf = FIGS_DIR / f'Exp4-stock-join-{MAIN_PCT}pct.pdf'
plt.savefig(str(out_pdf), format='pdf')
plt.show()
print(f'Saved: {out_pdf}')

In [ ]:
# ── Figure 2 (optional): Total latency vs update intensity ─────────────────
#    Line chart: SNAP vs EPOCH-WR (best) vs MONO-NR (worst MVHT)

HIGHLIGHT = [
    ('naive', '',   'SNAP',     '#EE6677', '-',  'o'),
    ('par',   'WR', 'EPOCH-WR', '#4477AA', '-',  's'),
    ('heap',  'NR', 'MONO-NR',  '#CCBB44', '--', '^'),
]

fig2, ax2 = plt.subplots(figsize=(6.0, 4.5))
plt.rcParams['font.family'] = 'serif'

for (t, r, label, color, ls, marker) in HIGHLIGHT:
    rows = df[(df['table'] == t) & (df['repair'] == r)].sort_values('update_pct')
    if rows.empty:
        print(f'  No data for {label}')
        continue
    ax2.plot(
        rows['update_pct'].astype(float),
        rows['total_ms'],
        label=label, color=color, linestyle=ls, marker=marker,
        linewidth=1.8, markersize=6,
    )

ax2.set_xlabel('Update Intensity (%)')
ax2.set_ylabel('Total Latency (ms)')
ax2.set_title(f'Exp 4: Total Latency vs Update Intensity\n(ORDER_LINE ⋈ STOCK, W={W}, {DIST})')
ax2.set_xticks(UPDATE_PCTS)
ax2.yaxis.grid(True, linestyle='--', linewidth=0.6, alpha=0.6)
ax2.legend(framealpha=0.9, fontsize=9)

plt.tight_layout()
out_pdf2 = FIGS_DIR / f'Exp4-stock-total-latency-vs-pct.pdf'
plt.savefig(str(out_pdf2), format='pdf')
plt.show()
print(f'Saved: {out_pdf2}')